# AEGIS — Model B: MobileNetV3-Small Condition Classifier (normal / damaged / unknown)

Trains a 3-class condition classifier per `AEGIS_Today_Scope.md`.

**Dataset:** SDNET2018 (Kaggle-native) as a `normal`/`damaged` proxy source.

**Known gap, flagged rather than hidden:** SDNET2018 only has Cracked/Uncracked labels, i.e. it
supplies `damaged` and `normal` examples, but zero `unknown` examples. The `unknown` output class is
kept (per the frozen 3-class taxonomy — `unknown` is a valid label, not a bug) but with today's single
dataset the model has **no training signal for it** and will not learn to predict it. This is a data
availability gap, not something this notebook can fix by inventing `unknown` images.

**Multi-session note:** same `CONTINUE_FROM_WEIGHTS` pattern as the detector notebook — no built-in
"resume" reliance on a persisted run folder, since Kaggle sessions don't guarantee that.

**If the dataset is not attached, cells below raise a clear `RuntimeError` telling you exactly what to
attach — they do not silently skip or fabricate data.**


In [ ]:
# --- Environment setup ---
import sys, subprocess

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

# Kaggle's preinstalled PyTorch build is periodically upgraded to a newer CUDA toolchain that drops
# kernels for older GPU architectures (observed: Tesla P100 / sm_60 raising "no kernel image is
# available for execution on the device" on a stock Kaggle image -- PyTorch dropped Pascal/sm_60
# support starting at 2.8.0). Kaggle can assign a P100, T4 x2, or newer accelerator depending on
# availability, and we don't control which -- so reinstall the latest cu118 build that still ships
# sm_60 kernels (2.7.1, one release before the Pascal drop) BEFORE anything imports torch. Earlier
# attempts pinned torch==2.2.2, which also has sm_60 -- but predates PyTorch's NumPy 2.0 support
# (added in 2.3), and forcibly downgrading numpy to match it broke a dozen other packages in Kaggle's
# base image that are compiled against NumPy 2.x's ABI (opencv-python, jax, cupy, etc. all crash).
# 2.7.1 supports both sm_60 and NumPy 2.x, so no numpy downgrade is needed at all.
gpu_present = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).returncode == 0
if gpu_present:
    print("GPU detected, installing a broadly architecture-compatible PyTorch build...")
    pip_install("torch==2.7.1", "torchvision==0.22.1", "--index-url", "https://download.pytorch.org/whl/cu118")
else:
    print("No GPU detected -- training will run on CPU and will be slow.")

pip_install("timm", "scikit-learn", "onnx")

import os, json, glob, random, hashlib
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from PIL import Image
import torchvision.transforms as T
from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix

print("Setup OK. CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} (compute capability sm_{cap[0]}{cap[1]})")
    arch_str = f"sm_{cap[0]}{cap[1]}"
    if arch_str not in torch.cuda.get_arch_list():
        raise RuntimeError(
            f"Installed PyTorch does not include kernels for this GPU's architecture ({arch_str}). "
            f"Supported: {torch.cuda.get_arch_list()}. The pinned torch==2.7.1+cu118 install above should "
            "cover this -- if you still see this, Kaggle assigned a newer/older GPU than expected; "
            "check https://pytorch.org/get-started/locally/ for a wheel matching this GPU."
        )


In [ ]:
# --- Config ---
SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CLASSES = ["normal", "damaged", "unknown"]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}

IMG_SIZE = 224
EPOCHS_PER_SESSION = 40   # spec calls for 40+ per session; run this notebook multiple times to extend
BATCH_SIZE = 64
LR = 3e-4
CHECKPOINT_EVERY_N_EPOCHS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WORKING = Path("/kaggle/working")
CHECKPOINT_OUT_DIR = WORKING / "checkpoints"
CHECKPOINT_OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- CONTINUE_FROM_WEIGHTS pattern (same rationale as the detector notebook) ---
# After a session, publish CHECKPOINT_OUT_DIR/last.pt as a Kaggle Dataset, attach it next run, and
# point this at its path -- this notebook does NOT rely on any run folder persisting across sessions.
CONTINUE_FROM_WEIGHTS = None
# Example: CONTINUE_FROM_WEIGHTS = "/kaggle/input/aegis-classifier-ckpt/last.pt"

if CONTINUE_FROM_WEIGHTS is not None and not Path(CONTINUE_FROM_WEIGHTS).exists():
    raise RuntimeError(
        f"CONTINUE_FROM_WEIGHTS is set to {CONTINUE_FROM_WEIGHTS!r} but that file does not exist. "
        "Attach the checkpoint dataset as Input, or set CONTINUE_FROM_WEIGHTS = None to start fresh."
    )

print("Classes:", CLASS_TO_ID)
print("Device:", DEVICE)
print("CONTINUE_FROM_WEIGHTS:", CONTINUE_FROM_WEIGHTS)


In [ ]:
# --- Helpers ---

def require_path(path, instructions):
    p = Path(path)
    if not p.exists():
        raise RuntimeError(f"Required path not found: {path}\n\n{instructions}")
    return p


def find_input_dir(name_fragments, base="/kaggle/input"):
    base = Path(base)
    if not base.exists():
        return None
    for child in base.iterdir():
        low = child.name.lower()
        if all(f.lower() in low for f in name_fragments):
            return child
    return None


ALL_SAMPLES = []  # list of dicts: {path, label, source_group}
MANIFEST = []

print("Helpers ready")


## Section 1 — SDNET2018 (`normal` / `damaged`)

**Manual step required:** attach the Kaggle dataset `aniruddhsharma/structural-defects-network-concrete-crack-images` as an Input to this kernel (Add Input -> search "Structural Defects Network SDNET 2018"). Folder structure is `{Decks,Pavements,Walls}/{Cracked,Uncracked}/*.jpg`.

In [ ]:
# --- SDNET2018 ---
SDNET_DIR = find_input_dir(["sdnet"]) or find_input_dir(["structural-defects"])
if SDNET_DIR is None:
    raise RuntimeError(
        "SDNET2018 dataset not found under /kaggle/input. Attach it: on Kaggle, click 'Add Input', "
        "search for 'Structural Defects Network (SDNET) 2018' "
        "(dataset slug: aniruddhsharma/structural-defects-network-concrete-crack-images), and attach it "
        "to this kernel, then re-run this cell."
    )

COMPONENT_DIRS = ["Deck", "Pavement", "Wall"]  # SDNET2018 top-level component categories
# Different SDNET2018 re-uploads spell these differently (verified live: this Kaggle mirror uses
# "Cracked" / "Non-cracked", not the "Cracked" / "Uncracked" the official dataset docs use) --
# accept known variants rather than hard-coding one spelling.
LABEL_DIR_CANDIDATES = [
    ({"cracked"}, "damaged"),
    ({"uncracked", "non-cracked", "noncracked", "non cracked"}, "normal"),
]

sdnet_by_class = {"normal": 0, "damaged": 0}
sdnet_used = 0

# SDNET2018 top-level folders are plural (Decks/Pavements/Walls); search flexibly for either form.
component_roots = []
for comp in COMPONENT_DIRS:
    matches = [p for p in SDNET_DIR.rglob("*") if p.is_dir() and p.name.lower() in (comp.lower(), comp.lower() + "s")]
    component_roots.extend(matches)

if not component_roots:
    raise RuntimeError(
        f"SDNET2018 is attached at {SDNET_DIR} but no Deck/Pavement/Wall component folders were found "
        "inside it. Inspect the attached dataset's folder layout and adjust COMPONENT_DIRS above."
    )

for comp_root in component_roots:
    for candidate_names, cls in LABEL_DIR_CANDIDATES:
        label_dirs = [p for p in comp_root.iterdir() if p.is_dir() and p.name.lower() in candidate_names]
        for label_dir in label_dirs:
            # rglob, not glob: large re-uploads of SDNET2018 sometimes chunk the (much larger)
            # Uncracked set into nested subfolders instead of a flat directory of ~47k files.
            for img_path in label_dir.rglob("*.jpg"):
                # SDNET2018 filenames look like "7001-115.jpg" -- "7001" identifies the physical
                # structure/specimen. Group by that ID (plus component) for the source-aware split,
                # since crops from the same structure are visually correlated, not independent samples.
                structure_id = img_path.stem.split("-")[0]
                source_group = f"sdnet_{comp_root.name}_{structure_id}"
                ALL_SAMPLES.append({"path": img_path, "label": CLASS_TO_ID[cls], "source_group": source_group})
                sdnet_by_class[cls] += 1
                sdnet_used += 1

print("SDNET2018 images used:", sdnet_used, "| by class:", sdnet_by_class)
if sdnet_used == 0:
    raise RuntimeError(
        "SDNET2018 was found and its component folders exist, but zero .jpg images were collected. "
        "Check the attached dataset actually contains images under Cracked/Uncracked subfolders."
    )
if sdnet_by_class["normal"] == 0 or sdnet_by_class["damaged"] == 0:
    sample_dir_names = sorted({p.name for comp_root in component_roots for p in comp_root.iterdir() if p.is_dir()})
    raise RuntimeError(
        f"SDNET2018 loaded images for only one class: {sdnet_by_class}. Refusing to train a classifier "
        "that has never seen one of its two available classes. Actual subfolder names found under the "
        f"component directories: {sample_dir_names}. Update LABEL_DIR_CANDIDATES above to match them."
    )

print(
    "\n[NOTE] 'unknown' class has zero examples from SDNET2018 -- it is structurally present in the "
    "3-class output but the model cannot learn to predict it from today's only dataset. See the "
    "markdown cell at the top of this notebook."
)

MANIFEST.append({
    "dataset": "SDNET2018",
    "source": "kaggle.com/datasets/aniruddhsharma/structural-defects-network-concrete-crack-images "
              "(Dorafshan, Thomas & Maguire 2018)",
    "license": "Open dataset, see Kaggle listing for full terms",
    "class_mapping": {"Cracked": "damaged", "Non-cracked/Uncracked (mirror-dependent)": "normal", "(none available)": "unknown"},
    "images_used": sdnet_used,
    "note": "Zero 'unknown'-labeled examples available today; that output class has no training signal.",
})


## Section 2 — Source-aware train/val/test split

Splits by `source_group` (structure/component ID), never by random image shuffle, per scope section 4 non-negotiable #1.

In [ ]:
if not ALL_SAMPLES:
    raise RuntimeError("No samples collected -- cannot build a split. See errors above.")

groups = sorted(set(s["source_group"] for s in ALL_SAMPLES))
rng = random.Random(SEED)
rng.shuffle(groups)

n = len(groups)
n_test = max(1, round(n * 0.15))
n_val = max(1, round(n * 0.15))
test_groups = set(groups[:n_test])
val_groups = set(groups[n_test:n_test + n_val])

def split_of(group):
    if group in test_groups:
        return "test"
    if group in val_groups:
        return "val"
    return "train"

for s in ALL_SAMPLES:
    s["split"] = split_of(s["source_group"])

split_counts = {sp: {c: 0 for c in CLASSES} for sp in ("train", "val", "test")}
for s in ALL_SAMPLES:
    split_counts[s["split"]][CLASSES[s["label"]]] += 1

print("Class counts per split:")
for sp in ("train", "val", "test"):
    total = sum(split_counts[sp].values())
    print(" ", sp, split_counts[sp], "total:", total)
    if total == 0:
        raise RuntimeError(f"Split {sp!r} ended up with zero images -- adjust split ratios or dataset sources.")


## Section 3 — Dataset manifest (scope section 4, non-negotiable #3)

In [ ]:
manifest_path = WORKING / "dataset_manifest.json"
manifest_path.write_text(json.dumps(MANIFEST, indent=2, default=str))
for entry in MANIFEST:
    print(f"- {entry['dataset']}: {entry['images_used']} images | license: {entry['license']}")
print("\nFull manifest written to", manifest_path)


## Section 4 — PyTorch Dataset / DataLoaders

In [ ]:
class ConditionDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s["path"]).convert("RGB")
        return self.transform(img), s["label"]


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.15, contrast=0.15),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_samples = [s for s in ALL_SAMPLES if s["split"] == "train"]
val_samples = [s for s in ALL_SAMPLES if s["split"] == "val"]
test_samples = [s for s in ALL_SAMPLES if s["split"] == "test"]

train_loader = DataLoader(ConditionDataset(train_samples, train_tf), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(ConditionDataset(val_samples, eval_tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(ConditionDataset(test_samples, eval_tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"train={len(train_samples)} val={len(val_samples)} test={len(test_samples)}")


## Section 5 — Model, loss, optimizer (with `CONTINUE_FROM_WEIGHTS`)

In [ ]:
model = timm.create_model("mobilenetv3_small_100", pretrained=(CONTINUE_FROM_WEIGHTS is None), num_classes=len(CLASSES))

start_epoch = 0
if CONTINUE_FROM_WEIGHTS is not None:
    ckpt = torch.load(CONTINUE_FROM_WEIGHTS, map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"])
    start_epoch = ckpt.get("epoch", 0)
    print(f"Resumed from {CONTINUE_FROM_WEIGHTS} at epoch {start_epoch}")

model = model.to(DEVICE)

# Class-weighted loss: SDNET2018 is imbalanced (far more Uncracked than Cracked), and 'unknown' has
# zero examples today -- give it weight 0 so it contributes nothing to the loss rather than an
# undefined/misleading gradient signal on a class with no data.
class_counts = np.array([split_counts["train"][c] for c in CLASSES], dtype=np.float64)
class_weights = np.zeros_like(class_counts)
nonzero = class_counts > 0
class_weights[nonzero] = class_counts[nonzero].sum() / (nonzero.sum() * class_counts[nonzero])
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print("Class weights:", dict(zip(CLASSES, class_weights.tolist())))

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
if CONTINUE_FROM_WEIGHTS is not None and "optimizer_state_dict" in ckpt:
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])


## Section 6 — Training loop with multi-session checkpointing

In [ ]:
def save_checkpoint(epoch, tag):
    path = CHECKPOINT_OUT_DIR / f"{tag}.pt"
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "classes": CLASSES,
    }, path)
    return path


def run_epoch(loader, train_mode):
    model.train(train_mode)
    total_loss, n_batches = 0.0, 0
    all_preds, all_labels = [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(train_mode):
            logits = model(x)
            loss = criterion(logits, y)
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item()
        n_batches += 1
        all_preds.extend(logits.argmax(1).detach().cpu().tolist())
        all_labels.extend(y.detach().cpu().tolist())
    macro_f1 = f1_score(all_labels, all_preds, labels=list(range(len(CLASSES))), average="macro", zero_division=0)
    return total_loss / max(n_batches, 1), macro_f1


for epoch in range(start_epoch, start_epoch + EPOCHS_PER_SESSION):
    train_loss, train_f1 = run_epoch(train_loader, train_mode=True)
    val_loss, val_f1 = run_epoch(val_loader, train_mode=False)
    print(f"epoch {epoch+1}: train_loss={train_loss:.4f} train_macroF1={train_f1:.4f} "
          f"val_loss={val_loss:.4f} val_macroF1={val_f1:.4f}")

    if (epoch + 1) % CHECKPOINT_EVERY_N_EPOCHS == 0:
        save_checkpoint(epoch + 1, "last")

final_epoch = start_epoch + EPOCHS_PER_SESSION
save_checkpoint(final_epoch, "last")
print(
    "\nTo continue training in a NEW session: publish", CHECKPOINT_OUT_DIR,
    "as a Kaggle Dataset (e.g. aegis-classifier-ckpt), attach it as Input next run, "
    "and set CONTINUE_FROM_WEIGHTS to its last.pt path in the Config cell above."
)


## Section 7 — Evaluate on held-out test split (never used during training)

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        logits = model(x)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(y.tolist())

macro_f1 = f1_score(all_labels, all_preds, labels=list(range(len(CLASSES))), average="macro", zero_division=0)
precision, recall, f1_per_class, support = precision_recall_fscore_support(
    all_labels, all_preds, labels=list(range(len(CLASSES))), zero_division=0
)
cm = confusion_matrix(all_labels, all_preds, labels=list(range(len(CLASSES))))

eval_report = {
    "macro_f1": float(macro_f1),
    "per_class": {
        CLASSES[i]: {"precision": float(precision[i]), "recall": float(recall[i]),
                     "f1": float(f1_per_class[i]), "support": int(support[i])}
        for i in range(len(CLASSES))
    },
    "confusion_matrix": {"labels": CLASSES, "matrix": cm.tolist()},
}
print(json.dumps(eval_report, indent=2))
(WORKING / "classifier_eval_report.json").write_text(json.dumps(eval_report, indent=2))


## Section 8 — Export ONNX

In [ ]:
model.eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
onnx_path = WORKING / "aegis_condition_classifier.onnx"
torch.onnx.export(
    model, dummy, str(onnx_path),
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)
print("Exported ONNX model to:", onnx_path)
